# Day 3 · Polygons · Study B: What AI Drinks

Data centers on top of **baseline water stress** (WRI Aqueduct 4.0): how much of the local water supply is already being withdrawn each year.
Adds `bws_cat` and `water_stress` to the master table, so every later map can use them.

**Needs:** the Aqueduct 4.0 download unzipped into `data/raw/` (see README).

## 1 · Settings

In [ ]:
DAY      = 3
PROMPT   = "Polygons"
STUDY    = "B"
SLUG     = "polygons-water-ai"
NOTEBOOK = "day03_polygons_water_ai.ipynb"

TITLE     = "What AI Drinks"
HIGHLIGHT = "AI"
SUBTITLE  = None        # None = filled in from the numbers in step 3
ABOUT     = ("Shaded areas are river basins where water is already under pressure, from WRI Aqueduct 4.0: "
             "the share of the available supply withdrawn each year by farms, industry and homes. "
             "Medium to high is 20 to 40%, high is 40 to 80%, extremely high is over 80%. "
             "Dots and circles are the same data centers and AI campuses as in Where AI Gets Built.")
CREDITS   = ["im3", "epoch", "aqueduct"]

REGION   = "us48"
TOOL     = "mapbox"
MAPBOX_STYLE = "dark"
PROJECTION   = "globe"

## 2 · Load

In [ ]:
import pandas as pd
import mapkit as mk
from mapkit import ai, publish

dcs = ai.clip(ai.master(), REGION)
ex, big = dcs[dcs.kind == "existing"], dcs[dcs.kind == "ai_campus"]
print(f"{len(ex):,} existing data centers · {len(big)} AI campuses")
water = ai.load_water_stress()          # first run trims the global file to the US (a minute or two)
print(f"{len(water):,} river basins with a water stress score")

## 3 · Analysis: which data centers sit where water is already scarce?

In [ ]:
dcs = ai.attach(dcs, water[["bws_cat", "water_stress", "geometry"]], ["bws_cat", "water_stress"])
ai.update_master(dcs, ["bws_cat", "water_stress"], about="Baseline annual water stress of the river basin",
                 source="WRI Aqueduct 4.0", day=DAY)

ex, big = dcs[dcs.kind == "existing"], dcs[dcs.kind == "ai_campus"]
coming = big[big.status != "Operating"]
high = lambda d: (d["bws_cat"] >= 3)
pct = lambda d: round(100 * high(d).sum() / max(len(d), 1))

print(f"Existing data centers in high or extremely high water stress: {high(ex).sum():,} of {len(ex):,} ({pct(ex)}%)")
print(f"AI campuses in high or extremely high water stress:          {high(big).sum()} of {len(big)} ({pct(big)}%)")
print(f"AI campuses still coming, in high stress:                   {high(coming).sum()} of {len(coming)}")
print("\nAI campuses by water stress:")
print(big["water_stress"].fillna("No data").value_counts().to_string())

if SUBTITLE is None:
    SUBTITLE = f"{pct(big)}% of the new AI campuses sit where water is already scarce"
print("\nSubtitle:", SUBTITLE)

# today's overlay: only basins at medium-high stress or worse, so the map stays readable
stress_order = ["Medium to high", "High", "Extremely high"]
stress_colors = dict(zip(stress_order, ["#16646a", "#2fb5a8", "#8fe3d6"]))
overlays = [mk.Layer(water[water["bws_cat"] >= 2], "Water stress", column="water_stress", order=stress_order,
                     colors=stress_colors, opacity=0.55, tooltip="water_stress", popup={"water_stress": "Water stress"})]
extra_popup = {"water_stress": "Water stress"}

## 4 · Map

In [ ]:
layers = overlays + ai.base_layers(dcs, extra_popup=extra_popup)    # overlay underneath, the cast on top
bounds = ai.REGIONS[REGION] if isinstance(REGION, str) else None
credits = mk.style.credits(*CREDITS)

if TOOL == "mapbox":
    m = mk.mapbox(layers, title=TITLE, subtitle=SUBTITLE, about=ABOUT, sources=credits, series="ai",
                  bounds=bounds, style=MAPBOX_STYLE, projection=PROJECTION, highlight=HIGHLIGHT)
elif TOOL == "folium":
    m = mk.interactive(layers, day=DAY, prompt=PROMPT, title=TITLE, subtitle=SUBTITLE, sources=credits,
                       series="ai", bounds=bounds)
else:
    m = mk.static(layers, day=DAY, prompt=PROMPT, title=TITLE, subtitle=SUBTITLE, sources=credits,
                  series="ai", bounds=bounds, crs="EPSG:5070")
m if TOOL == "folium" else None

## 5 · Export + caption numbers

In [ ]:
publish.save(m, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE, subtitle=SUBTITLE, series="ai", study=STUDY,
             tool={"mapbox": "Python + Mapbox", "folium": "Python + Folium", "static": "Python"}[TOOL],
             notebook=NOTEBOOK, credits=mk.style.credits(*CREDITS, full=True))

print(f"""
CAPTION NUMBERS
- {high(big).sum()} of {len(big)} AI campuses ({pct(big)}%) sit in high or extremely high water stress
- {high(coming).sum()} of the {len(coming)} still being built or expanding
- existing data centers: {pct(ex)}% in high or extremely high stress
""")